# Synthetic Dataset Generator (profe-ssor)

Generate datasets using **OpenAI** and **OpenRouter**, with **mixed prompt styles** for diverse output.

**Choices:**
- **Models:** OpenAI + OpenRouter (local)
- **Prompts:** Mixed (vary style per row)
- **Schemas:** Fixed (reviews, tickets, Q&A) + Custom
- **Generation:** One big LLM call for N rows

Fill in the code cells below step by step.

---
## Step 1: Imports

Import what you need: `os`, `json`, `dotenv`, `OpenAI`, `gradio`, and anything else for building prompts or parsing the model output.

In [1]:
# Your code here
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json




---
## Step 2: Initialise clients (OpenAI + OpenRouter)

1. Call `load_dotenv(override=True)`.
2. Read `OPENROUTER_API_KEY` from the environment.
3. If the key exists, create an OpenRouter client (`OpenAI(api_key=..., base_url="https://openrouter.ai/api/v1")`) and set a model id (e.g. `openai/gpt-4o-mini` or a Claude model).
4. Else create a normal OpenAI client and set an OpenAI model (e.g. `gpt-4o-mini`).
5. Keep one variable for the **client** and one for the **model string** so the rest of the notebook can call "the chosen API" with one client + one model.

In [ ]:
load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Ollama (local) - OpenAI-compatible API at localhost:11434
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
OLLAMA_MODEL = "llama3.2"

# OpenRouter (when key is set)
if openrouter_api_key:
    openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
    OPENROUTER_MODEL = "openai/gpt-4o-mini"
else:
    openrouter_client = None
    OPENROUTER_MODEL = None

print("Ollama:", OLLAMA_MODEL)
print("OpenRouter:", "ready" if openrouter_api_key else "no key")


Ollama: llama3.2
OpenRouter: ready


---
## Step 3: Define schemas (fixed + custom)

1. **Fixed schemas:** Define three schema presets (e.g. dicts or dataclasses):
   - **Product reviews:** e.g. `title`, `rating` (1–5), `body`, `date`.
   - **Support tickets:** e.g. `ticket_id`, `topic`, `message`, `priority` (low/medium/high).
   - **Quiz Q&A:** e.g. `question`, `answer`, `difficulty` (easy/medium/hard).
2. For each, store a **short description** and **field names** (and optionally types) so you can build a prompt from them.
3. **Custom:** Allow a free-form text description (e.g. "Customer name, email, signup date"). When the user picks "Custom", use that text as the schema description in the prompt.

Implement a small helper that, given schema choice + optional custom text, returns the **string** you will paste into the LLM prompt (e.g. "Output JSON with keys: title, rating, body, date").

In [ ]:
SCHEMAS = {
    "Product reviews": {
        "description": "Product reviews with title, 1-5 rating, body text, and date.",
        "fields": ["title", "rating", "body", "date"],
        "hint": "rating is integer 1-5; date is YYYY-MM-DD.",
    },
    "Support tickets": {
        "description": "Support tickets with id, topic, message, and priority.",
        "fields": ["ticket_id", "topic", "message", "priority"],
        "hint": "priority is one of: low, medium, high.",
    },
    "Quiz Q&A": {
        "description": "Quiz questions and answers with difficulty.",
        "fields": ["question", "answer", "difficulty"],
        "hint": "difficulty is one of: easy, medium, hard.",
    },
}


def get_schema_description(schema_choice, custom_text=""):
    """
    Given schema_choice (dropdown value) and optional custom_text, return a single
    string describing the schema for the prompt.
    """
    if schema_choice == "Custom" and custom_text and custom_text.strip():
        return custom_text.strip()
    if schema_choice not in SCHEMAS:
        return "JSON objects with any consistent keys."
    s = SCHEMAS[schema_choice]
    fields_str = ", ".join(s["fields"])
    return f"Output JSON with keys: {fields_str}. {s.get('hint', '')}"

def get_schema_description_verbose(schema_choice, custom_text=""):
    """Same as above but returns a longer description (e.g. for strict formats)."""
    if schema_choice == "Custom" and custom_text and custom_text.strip():
        return f"Use exactly these fields (describe as you like): {custom_text.strip()}"
    if schema_choice not in SCHEMAS:
        return "JSON array of objects; keep keys consistent across items."
    s = SCHEMAS[schema_choice]
    parts = [f"Keys: {', '.join(s['fields'])}.", s["description"]]
    if s.get("hint"):
        parts.append(s["hint"])
    return " ".join(parts)

print(get_schema_description("Product reviews"))
print("---")
print(get_schema_description("Custom", "Customer name, email, signup date"))


Output JSON with keys: title, rating, body, date. rating is integer 1-5; date is YYYY-MM-DD.
---
Customer name, email, signup date


---
## Step 4: Build the prompt (Mixed styles)

1. **System prompt:** Tell the model it must output **only** a JSON array of N objects, each matching the schema. Say that it should **vary** tone, length, and style across rows (Mixed) so the dataset is diverse.
2. **User prompt:** Include:
   - The schema description (from Step 3).
   - The number of rows N.
   - A reminder to vary style (formal, casual, technical) across items.
3. Optional: add 1–2 short examples of desired JSON shape in the system or user prompt so the model stays on format.

Write a function, e.g. `build_prompts(schema_description, num_rows)`, that returns `(system_prompt, user_prompt)`.

In [4]:
def build_prompts(schema_description, number_rows):
    system_prompt = """You are a helpful assistant that generates synthetic data. You must output ONLY a single JSON array of objects—no markdown, no extra text. Vary tone, length, and style across rows (e.g. mix formal, casual, and technical) so the dataset is diverse."""
    user_prompt = f"""Generate exactly {number_rows} items as a JSON array. Schema: {schema_description}. Vary style across items (formal, casual, technical) and keep each object on one line if possible. Output only the JSON array, e.g. [{{"key": "value"}}, ...]."""
    return system_prompt, user_prompt



---
## Step 5: One big LLM call for N rows

1. Write a function `generate_dataset(schema_choice, custom_description, num_rows, client, model)` that:
   - Resolves the schema (fixed or custom) and gets the schema description string.
   - Calls `build_prompts(schema_description, num_rows)`.
   - Sends **one** `chat.completions.create` call with `system` and `user` messages; use the model and client from Step 2.
   - Asks for JSON only (e.g. `response_format="json_object"` if available, or instruct in the prompt).
2. Parse the response content as JSON (array of objects).
3. If parsing fails, return an empty list or a clear error message so the UI can show it.
4. Return the list of objects (and optionally the raw string for debugging).

In [ ]:
def generate_dataset(schema_choice, custom_description, num_rows, client, model):
    """
    One LLM call to generate N rows. Returns (list of objects, error_message or None).
    """

    schema_description = get_schema_description(schema_choice, custom_description or "")

    system_prompt, user_prompt = build_prompts(schema_description, num_rows)

    try:
        kwargs = {"model": model, "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]}
        if "openai" in model or "gpt" in model:
            kwargs["response_format"] = {"type": "json_object"}
        response = client.chat.completions.create(**kwargs)
        content = (response.choices[0].message.content or "").strip()
    except Exception as e:
        err_msg = str(e)
        if "429" in err_msg or "quota" in err_msg.lower() or "insufficient_quota" in err_msg.lower():
            return [], "OpenAI quota exceeded. Use Model: OpenRouter in the dropdown, or add payment at https://platform.openai.com"
        return [], err_msg

    if "```" in content:
        for start in ("```json", "```"):
            if content.strip().startswith(start):
                content = content.strip()[len(start):].strip()
                break
        if content.endswith("```"):
            content = content[:-3].strip()

    def extract_first_json(s):
        s = s.strip()
        for start, end in (("[", "]"), ("{", "}")):
            i = s.find(start)
            if i == -1:
                continue
            depth = 0
            for j in range(i, len(s)):
                if s[j] == start:
                    depth += 1
                elif s[j] == end:
                    depth -= 1
                    if depth == 0:
                        return json.loads(s[i : j + 1])
        return None

    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        data = extract_first_json(content)
        if data is None:
            return [], f"Invalid JSON: could not parse array or object"

    if isinstance(data, list):
        return data, None
    if isinstance(data, dict):
        if all(not isinstance(v, list) for v in data.values()):
            return [data], None
        # Object with array inside: find the array
        for key in ("items", "data", "rows", "records", "array", "results", "output", "list"):
            if key in data and isinstance(data[key], list):
                return data[key], None
        for v in data.values():
            if isinstance(v, list):
                return v, None

    return [], "Response was not a JSON array or object with array"


---
## Step 6: Gradio UI

1. **Schema:** Dropdown with options: "Product reviews", "Support tickets", "Quiz Q&A", "Custom".
2. **Custom description:** Textbox (visible when "Custom" is selected; optional: use `gr.update(visible=True/False)`).
3. **Model:** Dropdown (e.g. "OpenAI (gpt-4o-mini)", "OpenRouter (openai/gpt-4o-mini)" or another OpenRouter model). Wire this to the client/model you set in Step 2 (e.g. re-read env and choose client inside the submit function, or pass two functions).
4. **Number of rows:** Slider or number input (e.g. 5–50).
5. **Generate** button that calls your `generate_dataset` and shows progress (e.g. `gr.Progress()`).
6. **Output:**
   - Preview: a **Dataframe** or **JSON** or **Markdown** table of the generated rows (first 10–20 is enough).
   - **Download:** provide the full dataset as CSV and/or JSON (e.g. `gr.DownloadButton` or return a file path).

Wire the Generate button to a function that: gets schema + custom text + num_rows + model choice, calls `generate_dataset`, then returns preview data and a path to a temp CSV/JSON file for download.

In [6]:
import tempfile
import csv

SCHEMA_OPTIONS = ["Product reviews", "Support tickets", "Quiz Q&A", "Custom"]
MODEL_OPTIONS = ["Ollama (llama3.2)", "OpenRouter (openai/gpt-4o-mini)"]

def get_client_and_model(model_choice):
    """Return (client, model) for the chosen option."""
    if model_choice == "OpenRouter (openai/gpt-4o-mini)" and openrouter_client is not None:
        return openrouter_client, OPENROUTER_MODEL
    return ollama_client, OLLAMA_MODEL

def run_generate(schema_choice, custom_text, num_rows, model_choice, progress=gr.Progress()):
    progress(0.1, desc="Preparing...")
    client, model = get_client_and_model(model_choice)
    progress(0.3, desc="Generating...")
    data, err = generate_dataset(schema_choice, custom_text or "", num_rows, client, model)
    if err:
        return {"headers": [], "data": []}, err, None
    progress(0.9, desc="Preparing download...")
    # Preview: first 20 rows (gr.Dataframe can take dict with headers + data, or list of lists)
    if not data:
        return {"headers": [], "data": []}, "No rows generated.", None
    keys = list(data[0].keys()) if data else []
    preview_data = [[row.get(k, "") for k in keys] for row in data[:20]]
    preview = {"headers": keys, "data": preview_data}
    # Save full dataset to temp CSV for download
    fd, path = tempfile.mkstemp(suffix=".csv")
    with os.fdopen(fd, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(data)
    progress(1.0)
    return preview, "", path

with gr.Blocks(title="Synthetic Dataset Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Generate synthetic data: pick schema, model, and number of rows.")
    with gr.Row():
        schema_dd = gr.Dropdown(SCHEMA_OPTIONS, value="Product reviews", label="Schema")
        model_dd = gr.Dropdown(MODEL_OPTIONS, value=MODEL_OPTIONS[1] if openrouter_client else MODEL_OPTIONS[0], label="Model")
    custom_tb = gr.Textbox(placeholder="e.g. Customer name, email, signup date", label="Custom schema (only for Custom)", visible=False)
    num_rows = gr.Slider(5, 50, value=10, step=1, label="Number of rows")
    gen_btn = gr.Button("Generate", variant="primary")

    err_out = gr.Textbox(label="Status / Error", interactive=False)
    preview_df = gr.Dataframe(label="Preview (first 20 rows)", headers=None)
    download_file = gr.File(label="Download CSV")

    def toggle_custom(schema):
        return gr.update(visible=(schema == "Custom"))

    schema_dd.change(toggle_custom, schema_dd, custom_tb)

    gen_btn.click(
        fn=run_generate,
        inputs=[schema_dd, custom_tb, num_rows, model_dd],
        outputs=[preview_df, err_out, download_file],
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
